# Gestión de vacaciones para autolavados: documentación

Este notebook documenta **cómo se construyó la aplicación, paso a paso**, y **demuestra cada funcionalidad ejecutando el código real** de `app.py`.

**Las pruebas no tocan tus datos.** Usan una base de datos temporal, y los emails se capturan en una lista en lugar de enviarse.

### Cómo ejecutarlo
1. Abre este notebook en VS Code.
2. Arriba a la derecha, en *Select Kernel*, elige el Python del `venv` (`..\venv\Scripts\python.exe`).
3. Si te pide instalar `ipykernel`, acepta. También puedes instalarlo a mano: `..\venv\Scripts\python.exe -m pip install ipykernel`.
4. Pulsa **Run All**.

El resumen completo (instalación, configuración del correo, API, problemas frecuentes) está en `README.md`.

## Índice

| Paso | Qué se hizo |
|---|---|
| 0 | Código inicial |
| 1 | Revisión y problemas detectados |
| 2 | Primera versión completa: formulario, panel y validaciones |
| 3 | Incidencia: `ERR_CONNECTION_REFUSED` |
| 4 | Varios autolavados, emails y migración de la base de datos |
| 5 | Gestión de empleados e identificación por email |
| 6 | Incidencia: la solicitud no se enviaba |
| 7 | Días no disponibles sombreados en el calendario |
| 8 | Incidencia: los correos no llegaban |
| 9 | Documentación (`README.md` y este notebook) |
| 10 | Repositorio git propio, separado del Escritorio |
| 11 | Configuración real del correo con Gmail |
| 12 | Formulario de solicitud más profesional |
| 13 | Panel *Mis vacaciones* y días de vacaciones al año |
| 14 | Días no disponibles en el panel |
| 15 | Contraseña del panel de gestión |
| 16 | Versión para incrustar en la web del negocio (GoDaddy) |

## Paso 0: Código inicial

El punto de partida fue este `app.py`. Tenía una tabla `solicitudes` y un endpoint que devolvía los días ocupados en el formato que entiende la opción `disable` de flatpickr:

```python
@app.route("/api/ocupadas")
def ocupadas():
    rows = get_db().execute(
        "SELECT fecha_inicio, fecha_fin FROM solicitudes "
        "WHERE estado IN ('Pendiente', 'Aprobada')"
    ).fetchall()
    return jsonify([{"from": r["fecha_inicio"], "to": r["fecha_fin"]} for r in rows])

if __name__ == "__main__":
    init_db()
    app.run(debug=True)
```

## Paso 1: Problemas detectados

1. `init_db()` estaba dentro de `if __name__ == "__main__"`, así que con `flask run` la tabla no se creaba.
2. Cualquier solicitud bloqueaba las fechas para **todos** los empleados.
3. No se podían crear ni gestionar solicitudes: no había `POST`, ni panel, ni HTML.
4. El servidor no validaba los datos.

## Preparación del entorno de pruebas

Importamos la aplicación real y la apuntamos a una base de datos temporal. También sustituimos la función `enviar_email` por una que guarda los emails en una lista, para poder verlos sin enviarlos.

Como el panel de gestión está protegido (paso 15), el cliente de pruebas inicia sesión con una contraseña de demostración. No es la contraseña real del `.env`.

Las fechas se calculan a partir de hoy (`dia(n)` = hoy + n días), así el notebook sigue funcionando en el futuro.

In [ ]:
import os, re, sys, sqlite3, tempfile
from datetime import date, timedelta

sys.path.insert(0, os.getcwd())
import app as m

# Base de datos temporal: tus datos reales no se tocan
carpeta_tmp = tempfile.mkdtemp()
m.DB = os.path.join(carpeta_tmp, "pruebas.db")
m.init_db()

# Capturar emails en vez de enviarlos
emails = []
m.enviar_email = lambda destino, asunto, cuerpo: emails.append(
    {"para": destino, "asunto": asunto, "cuerpo": cuerpo})

# El panel de gestión pide contraseña (paso 15): en las pruebas usamos una de demostración
m.ADMIN_PASSWORD = "demo"
cliente = m.app.test_client()
cliente.post("/admin/login", data={"password": "demo"})

def dia(n):
    """Fecha ISO de dentro de n días."""
    return (date.today() + timedelta(days=n)).isoformat()

def ver(resp):
    """Muestra el código HTTP y la respuesta JSON."""
    print(resp.status_code, resp.get_json())

print("Base de datos de pruebas:", m.DB)

## Paso 2: Primera versión completa

Se hizo lo siguiente:
- `init_db()` se ejecuta **al importar** el módulo, así que funciona con `python app.py` y con `flask run`.
- Se crearon los endpoints para crear, listar y resolver solicitudes.
- Se añadieron validaciones en el servidor: no basta con bloquear días en el calendario, porque alguien podría saltárselo.
- La comprobación de solapamiento y la inserción van dentro de una transacción `BEGIN IMMEDIATE`. Así, dos peticiones que lleguen a la vez no pueden reservar los mismos días.
- Se crearon las páginas `index.html` (formulario con flatpickr en modo rango y en español) y `admin.html` (panel).

Estas son las tablas que crea `init_db()` en la versión final, tras los pasos 4 y 5:

In [ ]:
con = sqlite3.connect(m.DB)
for (tabla,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"):
    columnas = [f"{c[1]} ({c[2]})" for c in con.execute(f"PRAGMA table_info({tabla})")]
    print(f"{tabla}:\n   " + "\n   ".join(columnas) + "\n")
con.close()

## Paso 4: Varios autolavados

**Requisito:** la amiga tiene varios autolavados. Las vacaciones solo deben chocar entre empleados **del mismo autolavado**.

Se creó la tabla `autolavados`, con el nombre y el email del encargado, y se gestiona desde el panel `/admin`. El nombre es único.

In [ ]:
ver(cliente.post("/api/autolavados", json={"nombre": "Centro", "email_encargado": "jefe.centro@ejemplo.com"}))
ver(cliente.post("/api/autolavados", json={"nombre": "Norte",  "email_encargado": "jefe.norte@ejemplo.com"}))

print("\nNombre repetido:")
ver(cliente.post("/api/autolavados", json={"nombre": "Centro", "email_encargado": "otro@ejemplo.com"}))

print("\nEmail no válido:")
ver(cliente.post("/api/autolavados", json={"nombre": "Sur", "email_encargado": "no-es-un-email"}))

LAVADOS = {a["nombre"]: a["id"] for a in cliente.get("/api/autolavados").get_json()}
LAVADOS

## Paso 5: Gestión de empleados

**Requisito:** que la administradora pueda dar de alta autolavados **y empleados** desde la web.

- Tabla `empleados` con nombre, email (único y guardado en minúsculas), autolavado y activo.
- El empleado **solo escribe su email**. El nombre y el autolavado salen de su ficha, así que no puede elegir otro autolavado ni pedir vacaciones si no está dado de alta.

In [ ]:
for nombre, email, lavado in [("Ana",  "Ana@Ejemplo.com",  "Centro"),
                              ("Luis", "luis@ejemplo.com", "Centro"),
                              ("Eva",  "eva@ejemplo.com",  "Norte")]:
    ver(cliente.post("/api/empleados", json={"nombre": nombre, "email": email, "autolavado_id": LAVADOS[lavado]}))

print("\nEmail repetido (aunque cambien las mayúsculas):")
ver(cliente.post("/api/empleados", json={"nombre": "Otra", "email": "ana@ejemplo.com", "autolavado_id": LAVADOS["Norte"]}))

print("\nLista de empleados:")
for e in cliente.get("/api/empleados").get_json():
    print(f"  {e['nombre']:5} {e['email']:18} {e['autolavado']:7} activo={e['activo']}")

Así se identifica el empleado en el formulario (`/api/identificar`). Da igual escribir el email con mayúsculas o con espacios:

In [ ]:
ver(cliente.get("/api/identificar?email=  ANA@ejemplo.com "))
print("\nEmail no dado de alta:")
ver(cliente.get("/api/identificar?email=nadie@ejemplo.com"))

## Validaciones al pedir vacaciones (paso 2, ampliado en el paso 5)

El servidor rechaza las solicitudes que no cumplen las reglas:

In [ ]:
def pedir(email, inicio, fin):
    return cliente.post("/api/solicitudes", json={"email": email, "fecha_inicio": inicio, "fecha_fin": fin})

print("Solicitud correcta:");             ver(pedir("ana@ejemplo.com",   dia(30), dia(34)))
print("\nFechas pasadas:");               ver(pedir("ana@ejemplo.com",   dia(-5), dia(-2)))
print("\nInicio posterior al fin:");      ver(pedir("ana@ejemplo.com",   dia(60), dia(55)))
print("\nFormato de fecha incorrecto:");  ver(pedir("ana@ejemplo.com",   "05/12/2026", "06/12/2026"))
print("\nEmail no dado de alta:");        ver(pedir("nadie@ejemplo.com", dia(70), dia(72)))

## Solapamiento solo dentro del mismo autolavado (paso 4)

Ana (Centro) ya tiene pedidos los días 30 a 34 desde hoy.
- **Luis**, también de Centro, no puede coger el día 32.
- **Eva**, de Norte, sí puede coger esos mismos días.

In [ ]:
print("Luis (Centro) intenta los días 32-36:"); ver(pedir("luis@ejemplo.com", dia(32), dia(36)))
print("\nEva (Norte) pide los días 32-36:");    ver(pedir("eva@ejemplo.com",  dia(32), dia(36)))
print("\nLuis (Centro) pide los días 40-42:");  ver(pedir("luis@ejemplo.com", dia(40), dia(42)))

## Emails al encargado (paso 4)

Cada solicitud nueva genera un email al **encargado de su autolavado**. El email incluye un enlace con un token aleatorio y único.

En la app real se envían por SMTP con la configuración del `.env`, en segundo plano para que el formulario no espere. Si no hay `.env`, se escriben en la consola. Aquí los hemos capturado en la lista `emails`:

In [ ]:
for e in emails:
    print(f"Para: {e['para']}\nAsunto: {e['asunto']}\n")
print("─" * 60)
print(emails[0]["cuerpo"])

## Aprobación desde el enlace del email (paso 4)

El enlace `/revisar/<token>` abre una página con los datos y los botones **Aprobar** y **Rechazar**.

La acción se hace con un **POST** (al pulsar el botón) y no al abrir el enlace. Así, los antivirus o las vistas previas del correo que abren enlaces automáticamente no pueden aprobar nada por error.

In [ ]:
enlace = re.search(r"/revisar/\S+", emails[0]["cuerpo"]).group()
print("Enlace:", enlace)

pagina = cliente.get(enlace)
print("Abrir el enlace →", pagina.status_code, "| ¿Muestra el botón Aprobar?", "Aprobar" in pagina.text)

n_antes = len(emails)
resp = cliente.post(enlace, data={"estado": "Aprobada", "comentario": "¡Que lo disfrutes!"})
print("Pulsar Aprobar  →", resp.status_code, "| ¿Ahora pone 'aprobada'?", "aprobada" in resp.text)

resp = cliente.post(enlace, data={"estado": "Rechazada"})
print("Intentar cambiarla otra vez →", "ya no está pendiente" in resp.text)

print("Token inventado →", cliente.get("/revisar/token-falso").status_code)

print("\nEmail que recibe el empleado:\n" + "─" * 60)
print(emails[n_antes]["asunto"] + "\n")
print(emails[n_antes]["cuerpo"])

También se puede resolver desde el panel `/admin`, que usa `POST /api/solicitudes/<id>/estado`. Vamos a rechazar la de Luis (días 40 a 42). Al rechazarla, sus días quedan libres otra vez.

In [ ]:
sol_luis = next(s for s in cliente.get("/api/solicitudes").get_json()
                if s["empleado"] == "Luis" and s["estado"] == "Pendiente")
ver(cliente.post(f"/api/solicitudes/{sol_luis['id']}/estado", json={"estado": "Rechazada", "comentario": "Hay mucho trabajo"}))

print("\nAna pide ahora los días 40-42, que han quedado libres:")
ver(pedir("ana@ejemplo.com", dia(40), dia(42)))

## Paso 7: Días no disponibles sombreados

`/api/ocupadas` devuelve los rangos ocupados **del autolavado del empleado**. Cada rango lleva `estado` y `propia`, que indica si es del propio empleado.

El calendario usa la opción `onDayCreate` de flatpickr para añadir una clase CSS a cada día:

| Clase CSS | Color | Significado |
|---|---|---|
| `ocupado` | gris tachado | lo ha cogido un compañero |
| `propio-pendiente` | naranja | es tuyo y está pendiente |
| `propio-aprobada` | verde | es tuyo y está aprobado |

Esto es lo que recibe el calendario de cada empleado:

In [ ]:
def calendario(email, lavado):
    r = cliente.get(f"/api/ocupadas?autolavado={LAVADOS[lavado]}&email={email}").get_json()
    for o in r:
        clase = ("propio-aprobada" if o["estado"] == "Aprobada" else "propio-pendiente") if o["propia"] else "ocupado"
        print(f"  {o['from']} → {o['to']}  {o['estado']:9}  propia={o['propia']!s:5}  → clase CSS: {clase}")

print("Calendario de Ana (Centro):")
calendario("ana@ejemplo.com", "Centro")

print("\nCalendario de Luis (Centro): los días de Ana le salen grises")
calendario("luis@ejemplo.com", "Centro")

print("\nCalendario de Eva (Norte): solo ve lo de su autolavado")
print(cliente.get(f"/api/ocupadas?autolavado={LAVADOS['Norte']}&email=eva@ejemplo.com").get_json())

## Paso 5 (continuación): bajas y cambios de autolavado

- Si se desmarca *Activo*, el empleado ya no puede pedir vacaciones, pero su historial se conserva.
- Si se cambia el autolavado de un empleado, sus solicitudes antiguas siguen en el autolavado donde se pidieron.

In [ ]:
emp = {e["email"]: e for e in cliente.get("/api/empleados").get_json()}

print("Dar de baja a Luis:")
ver(cliente.put(f"/api/empleados/{emp['luis@ejemplo.com']['id']}",
                json={"nombre": "Luis", "email": "luis@ejemplo.com", "autolavado_id": LAVADOS["Centro"], "activo": False}))
print("Luis intenta identificarse:"); ver(cliente.get("/api/identificar?email=luis@ejemplo.com"))

print("\nCambiar a Eva de Norte a Centro:")
ver(cliente.put(f"/api/empleados/{emp['eva@ejemplo.com']['id']}",
                json={"nombre": "Eva", "email": "eva@ejemplo.com", "autolavado_id": LAVADOS["Centro"]}))
print("Ahora Eva es de:", cliente.get("/api/identificar?email=eva@ejemplo.com").get_json()["autolavado"])
print("Su solicitud antigua sigue en:",
      [s["autolavado"] for s in cliente.get("/api/solicitudes").get_json() if s["empleado"] == "Eva"])

## Paso 4 (continuación): migración automática de la base de datos

Cuando se añadieron autolavados y empleados, ya existía un `vacaciones.db` con la estructura antigua y una solicitud dentro.

`init_db()` comprueba las columnas con `PRAGMA table_info` y añade las que faltan con `ALTER TABLE`, **sin perder datos**. Lo comprobamos con una base de datos que tiene la estructura original:

In [ ]:
bd_antigua = os.path.join(carpeta_tmp, "antigua.db")
con = sqlite3.connect(bd_antigua)
con.execute("""CREATE TABLE solicitudes (
    id INTEGER PRIMARY KEY AUTOINCREMENT, empleado TEXT NOT NULL, email TEXT NOT NULL,
    fecha_inicio TEXT NOT NULL, fecha_fin TEXT NOT NULL,
    estado TEXT NOT NULL DEFAULT 'Pendiente', comentario TEXT,
    creada TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP)""")
con.execute("INSERT INTO solicitudes (empleado, email, fecha_inicio, fecha_fin) VALUES ('Prueba', 'p@x.com', '2026-10-12', '2026-10-18')")
con.commit()
print("Antes: ", [c[1] for c in con.execute("PRAGMA table_info(solicitudes)")])
con.close()

db_real, m.DB = m.DB, bd_antigua
m.init_db()                       # aquí se hace la migración
m.DB = db_real

con = sqlite3.connect(bd_antigua)
print("Después:", [c[1] for c in con.execute("PRAGMA table_info(solicitudes)")])
print("Datos conservados:", con.execute("SELECT empleado, fecha_inicio, fecha_fin FROM solicitudes").fetchall())
con.close()

## Paso 10: Repositorio git propio

La carpeta del proyecto estaba dentro del repositorio git **del Escritorio**, que contiene documentos personales y está conectado a GitHub. Se creó un repositorio independiente solo para el proyecto:

```powershell
cd C:\Users\anoni\Desktop\Vacaciones\Vacaciones
git init -b main
git add .
git commit -m "Aplicación de gestión de vacaciones para autolavados"
```

El `.gitignore` deja fuera `.env` (contraseñas), `vacaciones.db` (datos reales), `__pycache__/` y `venv/`. Para subirlo a GitHub hay que crear un repositorio **vacío** en https://github.com/new y después ejecutar:

```powershell
git remote add origin https://github.com/USUARIO/REPO.git
git push -u origin main
```

## Paso 11: Configuración real del correo

Hubo dos problemas:
1. **Los datos se escribieron en `.env.ejemplo`.** La app solo lee `.env`, y `.env.ejemplo` se sube a git, así que la contraseña se habría publicado. Se pasaron a `.env`.
2. **Se usaba la contraseña normal de Gmail**, y Gmail cortaba la conexión. Se creó una **contraseña de aplicación** de 16 letras en https://myaccount.google.com/apppasswords.

Esta celda comprueba la configuración **sin mostrar datos privados**:

In [ ]:
from pathlib import Path
print("¿Existe .env?          ", Path(".env").exists())
print("¿.env ignorado por git?", ".env" in Path(".gitignore").read_text())
print("Servidor SMTP:         ", m.SMTP_HOST or "(sin configurar: los emails se escriben en la consola)")
print("¿Usuario configurado?  ", bool(m.SMTP_USER))
print("¿Contraseña con formato de contraseña de aplicación?",
      bool(m.SMTP_PASSWORD) and len(m.SMTP_PASSWORD.replace(" ", "")) == 16 and m.SMTP_PASSWORD.replace(" ", "").isalpha())

## Paso 12: Formulario de solicitud más profesional

Se rediseñó la página del empleado (`templates/index.html`):

- **Una sola tarjeta** con el título *Solicitar vacaciones* y el campo **Tu email** con el botón **Continuar** al lado. Pulsar *Enter* hace lo mismo.
- Al reconocer el email, aparecen **debajo** el nombre del empleado con su autolavado en una etiqueta y el selector de fechas.
- Se quitaron "Hola" y "¿No eres tú?". Si se cambia el email, el paso de fechas se oculta solo.
- **Fallo corregido:** las reglas CSS `display: flex` anulaban el atributo `hidden`, así que algunas secciones se veían antes de tiempo. Se añadió `[hidden] { display: none !important; }`.

## Paso 13: Panel "Mis vacaciones" y días al año

**Requisito:** un panel al lado del formulario con el calendario de los días pedidos, los días que le quedan a cada trabajador (por ejemplo, de 28) y una leyenda de colores.

Se hizo lo siguiente:
- **Base de datos:** columna `empleados.dias_anuales` (28 por defecto), añadida con migración automática.
- **`dias_por_anio(inicio, fin)`** reparte un rango entre años, por si cruza el fin de año.
- **`dias_usados(empleado_id, año)`** suma los días aprobados y pendientes.
- **Disponibles = días al año − aprobados − pendientes.** Las pendientes cuentan para que nadie pida dos veces los mismos días.
- **Límite:** al crear una solicitud no se puede superar lo disponible. Se comprueba dentro de la misma transacción que el solapamiento.
- **Endpoint nuevo `GET /api/resumen`** con los datos que usa el panel.
- **Panel lateral** con contadores, barra de progreso, calendario de 12 meses, leyenda, lista de solicitudes y selector de año.
- En `/admin`, columnas nuevas **Días/año** (editable) y **Disponibles**.
- Se cuentan **días naturales**. Falta confirmar con la administradora si deben ser laborables.

Primero, cómo se reparte un rango que cruza el fin de año:

In [ ]:
Y = date.today().year + 1      # año que viene, para no chocar con fechas pasadas
print(f"Del 29/12/{Y} al 03/01/{Y+1}:", m.dias_por_anio(f"{Y}-12-29", f"{Y+1}-01-03"))
print(f"Del 01/03/{Y} al 07/03/{Y}:", m.dias_por_anio(f"{Y}-03-01", f"{Y}-03-07"))

Damos de alta un autolavado nuevo con dos empleados: **Marta**, con los 28 días por defecto, y **Pablo**, a media jornada con 10 días. Después probamos el límite:

In [ ]:
LAVADOS["Sur"] = cliente.post("/api/autolavados", json={"nombre": "Sur", "email_encargado": "jefe.sur@ejemplo.com"}).get_json()["id"]
ver(cliente.post("/api/empleados", json={"nombre": "Marta", "email": "marta@ejemplo.com", "autolavado_id": LAVADOS["Sur"]}))
ver(cliente.post("/api/empleados", json={"nombre": "Pablo", "email": "pablo@ejemplo.com", "autolavado_id": LAVADOS["Sur"], "dias_anuales": 10}))

print("\nDías al año fuera de rango:")
ver(cliente.post("/api/empleados", json={"nombre": "X", "email": "x@ejemplo.com", "autolavado_id": LAVADOS["Sur"], "dias_anuales": 500}))

print(f"\nPablo pide 7 días (01-07/03/{Y}):");          ver(pedir("pablo@ejemplo.com", f"{Y}-03-01", f"{Y}-03-07"))
print(f"\nPablo pide 4 más, pero solo le quedan 3:");    ver(pedir("pablo@ejemplo.com", f"{Y}-05-01", f"{Y}-05-04"))
print(f"\nPablo pide 3 (01-03/05/{Y}):");                ver(pedir("pablo@ejemplo.com", f"{Y}-05-01", f"{Y}-05-03"))

El panel *Mis vacaciones* recibe esto de `/api/resumen`. Aprobamos la primera solicitud de Pablo y rechazamos la segunda: los días rechazados vuelven a estar disponibles.

In [ ]:
def resumen(email, anio):
    r = cliente.get(f"/api/resumen?email={email}&anio={anio}").get_json()
    print(f"  {r['anio']}: {r['dias_anuales']} al año | aprobados {r['aprobados']} | "
          f"pendientes {r['pendientes']} | DISPONIBLES {r['disponibles']}")
    for s in r["solicitudes"]:
        print(f"     {s['from']} → {s['to']}  {s['dias']} días  {s['estado']}")
    return r

print("Antes de resolver:")
r = resumen("pablo@ejemplo.com", Y)

cliente.post(f"/api/solicitudes/{r['solicitudes'][0]['id']}/estado", json={"estado": "Aprobada"})
cliente.post(f"/api/solicitudes/{r['solicitudes'][1]['id']}/estado", json={"estado": "Rechazada", "comentario": "Coincide con inventario"})
print("\nDespués de aprobar 7 y rechazar 3:")
resumen("pablo@ejemplo.com", Y);

Una solicitud que cruza el fin de año descuenta días de los dos años:

In [ ]:
ver(pedir("marta@ejemplo.com", f"{Y}-12-29", f"{Y+1}-01-03"))
resumen("marta@ejemplo.com", Y)
resumen("marta@ejemplo.com", Y + 1);

print("\nEn el panel de administración (columna Disponibles, año actual):")
for e in cliente.get("/api/empleados").get_json():
    if e["autolavado"] == "Sur":
        print(f"  {e['nombre']:6} {e['disponibles']} de {e['dias_anuales']}")

## Paso 14: Días no disponibles en el panel

**Requisito:** que el trabajador vea qué días no puede coger porque ya los tiene un compañero.

- El calendario anual del panel pinta en **gris con rayas** los días que tienen compañeros del mismo autolavado (pendientes o aprobados), y en **gris claro** los días que ya han pasado.
- Si un día es a la vez del empleado y de un compañero, se ve el estado del empleado.
- Sección nueva **Días no disponibles** con la lista de fechas futuras bloqueadas. **No muestra nombres** de compañeros, por privacidad.
- Se reutiliza `/api/ocupadas`: los rangos con `propia = False` son los que no puede coger.

Colores del calendario anual:

| Color | Significado |
|---|---|
| Verde | Tus vacaciones aprobadas |
| Naranja | Tus vacaciones pendientes |
| Rosa tachado | Tus solicitudes rechazadas |
| Gris con rayas | **No disponible**: lo tiene un compañero |
| Gris claro | Día ya pasado |
| Recuadro azul | Hoy |

Esto es lo que ve **Marta** en "Días no disponibles": los días de Pablo, sin su nombre.

In [ ]:
ocupadas = cliente.get(f"/api/ocupadas?autolavado={LAVADOS['Sur']}&email=marta@ejemplo.com").get_json()
hoy = date.today().isoformat()

print("Días no disponibles para Marta:")
for o in sorted(ocupadas, key=lambda o: o["from"]):
    if not o["propia"] and o["to"] >= hoy:
        print(f"  {o['from']} → {o['to']}  ({m.dias(o['from'], o['to'])} días)  No disponible")

print("\nSi Marta intenta coger uno de esos días:")
ver(pedir("marta@ejemplo.com", f"{Y}-03-05", f"{Y}-03-06"))

## Paso 15: Contraseña del panel de gestión

**Requisito:** proteger el panel antes de publicar la app. La administradora todavía no ha elegido contraseña, así que todo queda preparado para que solo tenga que escribirla en el `.env`:

```ini
ADMIN_PASSWORD=una-contraseña-larga
SECRET_KEY=64-caracteres-aleatorios      # firma la sesión
```

Se hizo lo siguiente:
- Pantalla de acceso `/admin/login` y salida `/admin/logout`. La sesión dura 12 horas.
- El decorador **`@solo_admin`** protege `/admin` y toda la API de gestión. Sin sesión, las páginas redirigen al acceso y la API responde `401`.
- La contraseña se compara con `hmac.compare_digest`, y cada fallo espera 1 segundo para frenar los intentos de adivinarla.
- **Sin contraseña configurada**, el panel solo se abre en local con `python app.py` (modo debug y desde `127.0.0.1`). Publicada, queda cerrada.
- El formulario del empleado, `/embed` y el enlace del encargado siguen siendo públicos.

Lo comprobamos con un cliente nuevo, que no ha iniciado sesión:

In [ ]:
import time
anonimo = m.app.test_client()

print("Sin sesión:")
print("  /admin           →", anonimo.get("/admin").status_code, "redirige a", anonimo.get("/admin").location)
for ruta in ["/api/empleados", "/api/solicitudes", "/api/autolavados"]:
    print(f"  {ruta:17}→", anonimo.get(ruta).status_code)
print("  aprobar solicitud→", anonimo.post("/api/solicitudes/1/estado", json={"estado": "Aprobada"}).status_code)

print("\nLo público sigue funcionando sin sesión:")
for ruta in ["/", "/embed", "/api/identificar?email=ana@ejemplo.com"]:
    print(f"  {ruta:40}→", anonimo.get(ruta).status_code)

t = time.time()
r = anonimo.post("/admin/login", data={"password": "mala"})
print(f"\nContraseña incorrecta → {r.status_code}, mensaje: {'Contraseña incorrecta' in r.text}, espera: {time.time() - t:.1f} s")
r = anonimo.post("/admin/login", data={"password": "demo"})
print("Contraseña correcta   →", r.status_code, "redirige a", r.location)
print("Ahora /api/empleados  →", anonimo.get("/api/empleados").status_code)
anonimo.get("/admin/logout")
print("Tras cerrar sesión    →", anonimo.get("/api/empleados").status_code)

Así se comporta el panel **si no hay contraseña configurada**. Simulamos que la app está publicada en internet: sin modo debug y con una petición que llega desde fuera.

In [ ]:
password_demo, m.ADMIN_PASSWORD = m.ADMIN_PASSWORD, None
m.app.debug = False
visitante = m.app.test_client()
visitante.environ_base["REMOTE_ADDR"] = "203.0.113.5"   # una IP de internet

r = visitante.get("/admin/login")
print("Publicada sin contraseña → /admin/login:", r.status_code, "| aviso de configurarla:", "ADMIN_PASSWORD" in r.text)
print("                           /api/empleados:", visitante.get("/api/empleados").status_code)

m.ADMIN_PASSWORD = password_demo   # volvemos a la configuración de las pruebas

## Paso 16: Versión para incrustar en la web del negocio

**Requisito:** la administradora quiere meter el formulario en la web del inventario, que está en **GoDaddy**.

Se hizo lo siguiente:
- **Ruta `/embed`:** la misma página que `/` pero sin menú, sin enlace al panel y con fondo transparente.
- **Altura automática:** la página envía su altura con `postMessage` para que el `<iframe>` crezca con el contenido.
- **`/admin` y `/revisar`** envían `X-Frame-Options: DENY`, así que ninguna web puede incrustarlos.

Código para pegar en GoDaddy (en *Websites + Marketing*, una sección **HTML**; en WordPress, un bloque **HTML personalizado**):

```html
<iframe id="vacaciones" src="https://DIRECCION-DE-LA-APP/embed"
        style="width:100%; height:1100px; border:0" title="Solicitar vacaciones"></iframe>
<script>
  window.addEventListener("message", function (e) {
    if (e.data && e.data.tipo === "vacaciones-altura") {
      document.getElementById("vacaciones").style.height = e.data.altura + "px";
    }
  });
</script>
```

**La app tiene que estar publicada con HTTPS.** Una web `https://` no puede incrustar `http://` ni `127.0.0.1`.

In [ ]:
normal, incrustada = cliente.get("/").text, cliente.get("/embed").text
print("Página normal (/):   menú:", "<nav>" in normal, "| enlace al panel:", "Panel de gestión" in normal)
print("Incrustada (/embed): menú:", "<nav>" in incrustada, "| enlace al panel:", "Panel de gestión" in incrustada,
      "| avisa de su altura:", "vacaciones-altura" in incrustada)

print("\n¿Quién se puede meter en un iframe? (X-Frame-Options)")
for ruta in ["/embed", "/admin", "/admin/login", "/revisar/cualquiera"]:
    print(f"  {ruta:20}", cliente.get(ruta).headers.get("X-Frame-Options") or "permitido")

## Incidencias resueltas

### Paso 3: `ERR_CONNECTION_REFUSED`
- **Causa:** el servidor no estaba arrancado.
- **Solución:** ejecutar `..\venv\Scripts\python.exe app.py` y dejar la terminal abierta mientras se usa la app.

### Paso 6: la solicitud no se enviaba
- **Causa:** el recargador automático de Flask (modo debug) se reinició a mitad de los cambios de código y se quedó ejecutando una versión a medias, que daba `NameError` en cada envío.
- **Solución:** se reinició el servidor.
- **Mejora:** el formulario ahora muestra *"Error del servidor. Inténtalo de nuevo."* si el servidor falla. Antes no mostraba nada.

### Paso 8: los correos no llegaban
- **Causa:** no existía el archivo `.env`, así que la app solo escribía los emails en la consola.
- **Solución:** crear `.env` a partir de `.env.ejemplo` con una contraseña de aplicación de Gmail y reiniciar el servidor. Ver la sección 4 del `README.md`.

## Estado final de las solicitudes

Esto es lo que vería la administradora en el panel `/admin` después de todas las pruebas:

In [ ]:
print(f"{'#':>2}  {'Autolavado':10} {'Empleado':8} {'Desde':10}  {'Hasta':10}  {'Estado':9}  Comentario")
for s in cliente.get("/api/solicitudes").get_json():
    print(f"{s['id']:>2}  {s['autolavado']:10} {s['empleado']:8} {s['fecha_inicio']}  {s['fecha_fin']}  "
          f"{s['estado']:9}  {s['comentario'] or ''}")

## Limpieza y siguientes pasos

La siguiente celda borra la base de datos temporal de las pruebas.

**Pendiente** :
- [ ] Elegir la contraseña del panel y escribirla en `ADMIN_PASSWORD` del `.env` (el código ya está preparado, paso 15).
- [x] ~~Configurar el correo en `.env`~~. Funciona; para producción, usar la cuenta del negocio.
- [ ] Publicar en PythonAnywhere y poner la dirección pública en `BASE_URL`.
- [ ] Subir a GitHub (paso 10).
- [ ] Confirmar días naturales o laborables con la administradora.
- [ ] Integrar en la web de inventario de GoDaddy con el `<iframe>` de `/embed` (paso 16). Falta saber qué producto de GoDaddy usan.

In [ ]:
import shutil
shutil.rmtree(carpeta_tmp, ignore_errors=True)
print("Base de datos temporal borrada.")